# Fase 2 — Limpieza y Preparación de Datos

Construye la tabla de features lista para modelar: **una fila por línea de crédito PAY** con su variable objetivo y todas las variables predictoras limpias, imputadas y normalizadas.

Las features de comportamiento crediticio se agregan a nivel de línea. Las features financieras (balance, income, Buró, etc.) son a nivel cliente y se unen a través de la cadena `Line → Person Id → RFC → profile_entity_id`.

## 1. Importaciones y rutas

In [ ]:
import pandas as pd
import numpy as np
import unicodedata, re
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date
from dateutil.relativedelta import relativedelta
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

ruta_archivo = "C:/Users/AnaMaríaRamírezLizár/PROYECTOS/data/Base de datos.xlsx"
ruta_output  = "C:/Users/AnaMaríaRamírezLizár/PROYECTOS/data/"

## 2. Carga de datos

In [ ]:
data_credito  = pd.read_excel(ruta_archivo, sheet_name='Creditos')
data_abonos   = pd.read_excel(ruta_archivo, sheet_name='Abonos')
data_clientes = pd.read_excel(ruta_archivo, sheet_name='Clientes')
data_balance  = pd.read_excel(ruta_archivo, sheet_name='balance_sheet')
data_income   = pd.read_excel(ruta_archivo, sheet_name='income_statement')
data_invoices = pd.read_excel(ruta_archivo, sheet_name='annual_invoices_comparisons')
data_employees= pd.read_excel(ruta_archivo, sheet_name='employees')
data_entity   = pd.read_excel(ruta_archivo, sheet_name='superia_entity')
data_score    = pd.read_excel(ruta_archivo, sheet_name='csv_score')
data_vc_pyme  = pd.read_excel(ruta_archivo, sheet_name='csv_vc_pyme')
data_registro = pd.read_excel(ruta_archivo, sheet_name='xls_registro')

print("Tablas cargadas correctamente.")

## 3. Reconstrucción de payc

Se replica la lógica del notebook 01 para obtener la tabla de cuotas con DPD y capital vencido por ministración PAY.

In [ ]:
# Personas morales y producto PAY
data_pm   = data_clientes.loc[data_clientes['Person Type'] == 'MORAL', 'Person Id']
dcred_pm  = data_credito[data_credito['Person Id'].isin(data_pm)]
data_ldc  = dcred_pm[(dcred_pm['Is line'] == 'SI') & (dcred_pm['Loan Status'] == 'AUTORIZADO')].copy()
data_ldc  = data_ldc.rename(columns={'Cosecha': 'Originacion'})
data_ldc['RFC'] = data_ldc['Person Id'].map(data_clientes.set_index('Person Id')['Taxpayer ID Number'])

data_min = dcred_pm[dcred_pm['Loan Type'] == 'MINISTRACION'].copy()
data_min['Originacion'] = data_min['Line'].map(data_ldc.set_index('Line')['Originacion'])
data_min['RFC']         = data_min['Person Id'].map(data_clientes.set_index('Person Id')['Taxpayer ID Number'])

pay  = data_min[data_min['Product'].str.contains('PAY')].copy()
payb = data_abonos[data_abonos['Product'].str.contains('PAY')].copy()
pay  = pay.rename(columns={'Period Number': 'Cuotas'})
pay['Capital cuota'] = (pay['Amount'] / pay['Cuotas']).round(2)
payb = payb.rename(columns={'Installment': 'Cuota'})
payb['Originacion'] = payb['Portfolio Id'].map(pay.set_index('Portfolio Id')['Originacion'])

pagos_agrup = (
    payb.groupby(['Portfolio Id', 'Cuota'])
    .agg({'Application Date': 'max', 'Capital': 'sum', 'Penalty': 'sum'})
    .reset_index()
)

pagos = pay.loc[pay.index.repeat(pay['Cuotas'])].assign(
    Cuota=lambda x: x.groupby('Portfolio Id').cumcount() + 1
)
# Amortización francesa para SUPERIA PAY
# K_k = C / (1+r)^(n-k+1)  donde C = P*r/(1-(1+r)^-n) y r = tasa anual / 12
mask_sp = pagos['Product'] == 'SUPERIA PAY'
r_sp = pagos.loc[mask_sp, 'Interest Rate'] / 12
n_sp = pagos.loc[mask_sp, 'Cuotas']
k_sp = pagos.loc[mask_sp, 'Cuota']
C_sp = pagos.loc[mask_sp, 'Amount'] * r_sp / (1 - (1 + r_sp)**(-n_sp))
pagos.loc[mask_sp, 'Capital cuota'] = (C_sp / (1 + r_sp)**(n_sp - k_sp + 1)).round(2)

payc = pagos.merge(pagos_agrup, on=['Portfolio Id', 'Cuota'], how='left')
payc['Application Date'] = np.where(payc['Product']=='PAY',np.where((payc['Capital']-payc['Capital cuota']).abs()<=1,pd.to_datetime(payc['Application Date']),pd.to_datetime(date.today())),pd.to_datetime(payc['Application Date']))
payc['Application Date'] = payc['Application Date'].fillna(pd.to_datetime(date.today()))
payc['Application Date'] = pd.to_datetime(payc['Application Date'])

def fcp(row):
    duedate = pd.to_datetime(row['Opening Date']) + relativedelta(months=row['Cuota'])
    return pd.to_datetime(duedate)

payc['Fecha vencimiento cuota'] = payc.apply(fcp, axis=1)
payc = payc[pd.to_datetime(payc['Fecha vencimiento cuota']) <= pd.to_datetime(date.today())]
payc['Moratorios'] = payc['Penalty'].fillna(0)

def dpdp(row):
    DPDUP = pd.to_numeric((row['Application Date'] - row['Fecha vencimiento cuota']).days, errors='coerce')
    if DPDUP < 0:
        return 0
    if row['Loan Status'] == 'LIQUIDADO':
        return DPDUP if row['Moratorios'] > 0 else 0
    if pd.to_datetime(row['Application Date']) != pd.to_datetime(date.today()):
        return DPDUP if row['Moratorios'] > 0 else 0
    return DPDUP

payc['DPD']   = payc.apply(dpdp, axis=1)
payc['mvenc'] = payc.apply(lambda r: r['Capital cuota'] if r['DPD'] > 0 else 0, axis=1)

print(f"payc construido: {payc.shape[0]:,} cuotas | {payc['Person Id'].nunique()} clientes PAY")

## 4. Variable objetivo

**Definición:** una línea de crédito se considera en incumplimiento si tuvo al menos una cuota con DPD > 30 en cualquiera de sus ministraciones.

`target = 1` → incumplimiento  
`target = 0` → cumplimiento

In [ ]:
target = (
    payc.groupby('Line')['DPD']
    .max()
    .reset_index()
    .rename(columns={'DPD': 'dpd_max'})
)
target['target'] = (target['dpd_max'] > 30).astype(int)

print("Distribución de la variable objetivo (nivel línea):")
print(target['target'].value_counts().rename({0: 'Cumplimiento (0)', 1: 'Incumplimiento (1)'}).to_string())
print(f"\nTasa de incumplimiento: {target['target'].mean():.1%}")
print(f"Total líneas: {len(target)}")

## 5. Tabla base de líneas PAY

Una fila por línea de crédito, con su `Person Id`, `RFC` y `profile_entity_id` para poder unir las features financieras.

In [ ]:
# Líneas únicas con Person Id y RFC (desde pay que ya está filtrado a PAY)
lineas_pay = pay[['Line', 'Person Id', 'RFC']].drop_duplicates(subset='Line')

# Agregar profile_entity_id via RFC → superia_entity
data_entity_sel = data_entity[['profile_entity_id', 'rfc']].drop_duplicates()
lineas_pay = lineas_pay.merge(
    data_entity_sel, left_on='RFC', right_on='rfc', how='left'
).drop(columns='rfc')

ids_pay_list  = lineas_pay['profile_entity_id'].dropna().unique().tolist()
rfcs_pay_list = lineas_pay['RFC'].dropna().unique().tolist()
rfc_a_pid = (
    lineas_pay.dropna(subset=['RFC', 'profile_entity_id'])
    .drop_duplicates('RFC')
    .set_index('RFC')['profile_entity_id'].to_dict()
)

print(f"Líneas PAY únicas:   {lineas_pay['Line'].nunique()}")
print(f"Clientes únicos:     {lineas_pay['Person Id'].nunique()}")
print(f"Con profile_entity_id: {lineas_pay['profile_entity_id'].notna().sum()}")

## 6. Construcción de features

### 6.1 Comportamiento crediticio por línea de crédito

In [ ]:
mob_col = 'MOB' if 'MOB' in payc.columns else 'Cuota'

feat_cred = (
    payc.groupby('Line').agg(
        n_ministraciones  = ('Portfolio Id', 'nunique'),
        monto_promedio    = ('Amount',        'mean'),
        monto_total       = ('Amount',        'sum'),
        cuotas_promedio   = ('Cuotas',        'mean'),
        tasa_interes      = ('Interest Rate',  'mean'),
        pct_cuotas_atraso = ('DPD',           lambda x: (x > 0).mean()),
        dpd_promedio      = ('DPD',           'mean'),
        dpd_max           = ('DPD',           'max'),
        mvenc_total       = ('mvenc',         'sum'),
        mob_max           = (mob_col,         'max'),
    ).reset_index()
)

feat_cred['pct_capital_vencido'] = np.where(
    feat_cred['monto_total'] > 0,
    feat_cred['mvenc_total'] / feat_cred['monto_total'],
    0
)

print(f"Features crediticias: {feat_cred.shape[1] - 1} variables | {len(feat_cred)} líneas")

### 6.2 Variables financieras — Balance e Income

Fuente primaria: `balance_sheet` e `income_statement`. Fallback: `xls_registro` para clientes sin datos en fuentes primarias.

In [ ]:
reg_pay = data_registro[data_registro['rfc'].isin(rfcs_pay_list)].copy()
reg_pay['profile_id'] = reg_pay['rfc'].map(rfc_a_pid)

# ── Balance ──────────────────────────────────────────────────────────────────
bal_primario = (
    data_balance[data_balance['profile_id'].isin(ids_pay_list)]
    .sort_values('year').groupby('profile_id').last()
    [['total_assets', 'total_liabilities', 'sh_equity',
      'current_assets', 'current_liabilities', 'current_debt', 'lt_debt']]
    .reset_index()
)
pids_con_balance = set(bal_primario['profile_id'])
bal_fallback = (
    reg_pay[~reg_pay['profile_id'].isin(pids_con_balance)]
    [['profile_id', 'u_total_assets', 'u_total_liabilities', 'u_sh_equity',
      'u_current_assets', 'u_current_liabilities', 'u_st_debt', 'u_lt_liabilities']]
    .rename(columns={
        'u_total_assets': 'total_assets', 'u_total_liabilities': 'total_liabilities',
        'u_sh_equity': 'sh_equity', 'u_current_assets': 'current_assets',
        'u_current_liabilities': 'current_liabilities',
        'u_st_debt': 'current_debt', 'u_lt_liabilities': 'lt_debt',
    })
)
feat_bal = pd.concat([bal_primario, bal_fallback], ignore_index=True)

# Ratios de balance (reducen multicolinealidad)
feat_bal['apalancamiento']    = feat_bal['total_liabilities'] / feat_bal['sh_equity'].replace(0, np.nan)
feat_bal['liquidez_corriente']= feat_bal['current_assets']   / feat_bal['current_liabilities'].replace(0, np.nan)
feat_bal['deuda_total']       = feat_bal['current_debt'] + feat_bal['lt_debt']

# ── Income ────────────────────────────────────────────────────────────────────
inc_primario = (
    data_income[data_income['profile_id'].isin(ids_pay_list)]
    .sort_values('year').groupby('profile_id').last()
    [['sales', 'gross_income', 'operating_income', 'net_income']]
    .reset_index()
)
pids_con_income = set(inc_primario['profile_id'])
inc_fallback = (
    reg_pay[~reg_pay['profile_id'].isin(pids_con_income)]
    [['profile_id', 'u_revenue', 'u_gross_profit', 'u_ebit', 'u_net_profit']]
    .rename(columns={
        'u_revenue': 'sales', 'u_gross_profit': 'gross_income',
        'u_ebit': 'operating_income', 'u_net_profit': 'net_income',
    })
)
feat_inc = pd.concat([inc_primario, inc_fallback], ignore_index=True)

# Ratios de income
feat_inc['margen_neto']    = feat_inc['net_income']       / feat_inc['sales'].replace(0, np.nan)
feat_inc['margen_bruto']   = feat_inc['gross_income']     / feat_inc['sales'].replace(0, np.nan)
feat_inc['margen_operativo']= feat_inc['operating_income']/ feat_inc['sales'].replace(0, np.nan)

print(f"Balance  — primaria: {len(pids_con_balance)} | fallback: {len(bal_fallback)} clientes")
print(f"Income   — primaria: {len(pids_con_income)} | fallback: {len(inc_fallback)} clientes")

### 6.3 Facturación SAT

In [ ]:
feat_inv = (
    data_invoices[data_invoices['profile_id'].isin(ids_pay_list)]
    .sort_values('period').groupby('profile_id').last()
    [['total_income', 'net_income', 'margin',
      'days_sales_outstanding', 'days_payable_outstanding']]
    .reset_index()
    .rename(columns={'net_income': 'sat_net_income', 'total_income': 'sat_total_income',
                     'margin': 'sat_margin'})
)

print(f"Facturación SAT: {len(feat_inv)} clientes")

### 6.4 Empleados

In [ ]:
feat_emp = (
    data_employees[data_employees['profile_id'].isin(ids_pay_list)]
    .sort_values(['year', 'month']).groupby('profile_id').last()
    [['total']]
    .reset_index()
    .rename(columns={'total': 'empleados'})
)

print(f"Empleados: {len(feat_emp)} clientes")

### 6.5 Score Buró (csv_score) y Variables Califica (csv_vc_pyme)

In [ ]:
def normalizar_nombre(nombre):
    if not isinstance(nombre, str):
        return ''
    nombre = unicodedata.normalize('NFD', nombre)
    nombre = ''.join(c for c in nombre if unicodedata.category(c) != 'Mn')
    nombre = nombre.upper()
    nombre = re.sub(r'[^\w\s]', '', nombre)
    sufijos = ['SA DE CV', 'S A DE C V', 'SAPI DE CV', 'SRL DE CV', 'SRL',
               'S DE RL DE CV', 'S DE PR DE RL DE CV', 'SC', 'AC', 'SA',
               'DE CV', 'SPR DE RL DE CV', 'SPR DE RL']
    for s in sufijos:
        nombre = re.sub(rf'\b{s}\b', '', nombre)
    return re.sub(r'\s+', ' ', nombre).strip()

# Nombres normalizados de clientes PAY
clientes_nombres = (
    data_clientes[data_clientes['Person Id'].isin(clientes_pay['Person Id'])]
    [['Person Id', 'Full Name Or Business Name']].copy()
)
clientes_nombres['nombre_norm'] = clientes_nombres['Full Name Or Business Name'].map(normalizar_nombre)
norm_a_pid = (
    clientes_nombres.merge(clientes_pay, on='Person Id', how='left')
    .set_index('nombre_norm')['profile_entity_id'].to_dict()
)

# Score Buró — último registro por empresa
data_score['nombre_norm'] = data_score['PRIMERNOMBRE'].map(normalizar_nombre)
feat_score = (
    data_score[data_score['nombre_norm'].isin(norm_a_pid)]
    .sort_values('FECHA CARGA').groupby('nombre_norm').last()
    [['VALORSCORE']].reset_index()
)
feat_score['profile_id'] = feat_score['nombre_norm'].map(norm_a_pid)
feat_score = feat_score[['profile_id', 'VALORSCORE']].rename(columns={'VALORSCORE': 'buro_score'})

# Variables califica Buró — último registro por empresa
data_vc_pyme['nombre_norm'] = data_vc_pyme['PRIMERNOMBRE'].map(normalizar_nombre)
feat_vc = (
    data_vc_pyme[data_vc_pyme['nombre_norm'].isin(norm_a_pid)]
    .sort_values('FECHA CARGA').groupby('nombre_norm').last()
    [['5', '6', '14']].reset_index()
)
feat_vc['profile_id'] = feat_vc['nombre_norm'].map(norm_a_pid)
feat_vc = feat_vc[['profile_id', '5', '6', '14']].rename(columns={
    '5': 'buro_pct_pago_bancario',
    '6': 'buro_pct_pago_no_bancario',
    '14': 'buro_quitas_12m'
})

print(f"Score Buró: {len(feat_score)} clientes")
print(f"Variables califica: {len(feat_vc)} clientes")

### 6.6 Variables de Buró desde xls_registro

In [ ]:
feat_reg = (
    reg_pay
    .sort_values('reg_date').groupby('profile_id').last()
    [['buro_active_loans', 'buro_active_amount', 'buro_late_loans',
      'buro_late_amount', 'loan_utilization', 'max_late_days',
      'operation_quantity', 'late_quantity', 'cub_score', 'cub_p_default']]
    .reset_index()
)

print(f"Variables xls_registro: {len(feat_reg)} clientes")

## 7. Unión en tabla maestra

In [ ]:
# Base: líneas PAY con target
df = lineas_pay.merge(target[['Line', 'target']], on='Line', how='left')

# Features crediticias (via Line)
df = df.merge(feat_cred, on='Line', how='left')

# Features financieras (via profile_entity_id — nivel cliente, se replica por línea)
df = df.merge(feat_bal,   left_on='profile_entity_id', right_on='profile_id', how='left').drop(columns='profile_id', errors='ignore')
df = df.merge(feat_inc,   left_on='profile_entity_id', right_on='profile_id', how='left').drop(columns='profile_id', errors='ignore')
df = df.merge(feat_inv,   left_on='profile_entity_id', right_on='profile_id', how='left').drop(columns='profile_id', errors='ignore')
df = df.merge(feat_emp,   left_on='profile_entity_id', right_on='profile_id', how='left').drop(columns='profile_id', errors='ignore')
df = df.merge(feat_score, left_on='profile_entity_id', right_on='profile_id', how='left').drop(columns='profile_id', errors='ignore')
df = df.merge(feat_vc,    left_on='profile_entity_id', right_on='profile_id', how='left').drop(columns='profile_id', errors='ignore')
df = df.merge(feat_reg,   left_on='profile_entity_id', right_on='profile_id', how='left').drop(columns='profile_id', errors='ignore')

print(f"Tabla maestra: {df.shape[0]} líneas | {df.shape[1]} columnas")
print(f"\nNulos por variable:")
nulos = df.isnull().sum()
nulos = nulos[nulos > 0]
pct   = (nulos / len(df) * 100).round(1)
print(pd.DataFrame({'nulos': nulos, '%': pct}).to_string())

## 8. Imputación con mediana

In [ ]:
# Columnas de identificación — no se imputan
cols_id = ['Line', 'Person Id', 'RFC', 'profile_entity_id', 'target']

cols_features = [c for c in df.columns if c not in cols_id]
medianas = df[cols_features].median()
df[cols_features] = df[cols_features].fillna(medianas)

print("Imputación completada.")
print(f"Nulos restantes: {df[cols_features].isnull().sum().sum()}")

## 9. Normalización (StandardScaler)

Se estandarizan las variables numéricas para que tengan media 0 y desviación estándar 1. Se excluyen variables binarias (`buro_quitas_12m`, `target`) y de identificación.

In [ ]:
cols_binarias = ['buro_quitas_12m']
cols_no_escalar = cols_id + cols_binarias
cols_escalar = [c for c in df.columns if c not in cols_no_escalar]

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[cols_escalar] = scaler.fit_transform(df[cols_escalar])

print(f"Variables normalizadas: {len(cols_escalar)}")
print(f"Variables sin normalizar: {cols_no_escalar}")

## 10. División train / test

Se usa estratificación por `target` para mantener la proporción de incumplimiento en ambos conjuntos. Split 80/20.

In [ ]:
X = df_scaled.drop(columns=cols_id + cols_binarias + ['target'])
y = df_scaled['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} líneas | Test: {len(X_test)} líneas")
print(f"Tasa incumplimiento — Train: {y_train.mean():.1%} | Test: {y_test.mean():.1%}")

## 11. Guardar datasets procesados

In [ ]:
# Tabla maestra completa (sin normalizar, con IDs)
df.to_csv(ruta_output + 'features_master.csv', index=False)

# Conjuntos de entrenamiento y prueba (normalizados, sin IDs)
X_train.to_csv(ruta_output + 'X_train.csv', index=False)
X_test.to_csv(ruta_output  + 'X_test.csv',  index=False)
y_train.to_csv(ruta_output + 'y_train.csv', index=False)
y_test.to_csv(ruta_output  + 'y_test.csv',  index=False)

print("Archivos guardados en data/:")
print("  features_master.csv — tabla maestra completa")
print("  X_train.csv / X_test.csv — features normalizadas")
print("  y_train.csv / y_test.csv — variable objetivo")